# GeoChange AI — STEP 2A.2: Reproducible Cleaning Manifest

This notebook runs `src/manifest.py`. It **does not modify the original Parquet files** and does not start the TensorFlow data pipeline.

It:
1. Counts JPEG/PNG/other signatures for every split and column.
2. Hashes encoded bytes for imageA, imageB and label.
3. Reports exact pair-key duplicates and partial image matches.
4. Applies the agreed train-only manifest rules.
5. Writes CSV reports under `data/processed/`.

The original validation and test rows are report-only and are never dropped.


In [ ]:
# 1. Mount Google Drive and set the project location

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path("/content/drive/MyDrive/GeoChange-AI")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = DATA_DIR / "processed"
SCRIPT_PATH = PROJECT_DIR / "src" / "manifest.py"

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Manifest script:", SCRIPT_PATH)

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Copy manifest.py into this location first: {SCRIPT_PATH}"
    )


In [ ]:
# 2. Run the manifest builder

# It reads the original Parquet files in batches and stores only hashes,
# paths and row metadata in memory. It does not decode images or change Parquet.

sys.path.insert(0, str(PROJECT_DIR / "src"))

from manifest import build_manifests

results = build_manifests(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR
)


In [ ]:
# 3. Inspect file-signature counts

signature_df = pd.DataFrame(results["signature_rows"])
display(signature_df)


In [ ]:
# 4. Inspect duplicate pair-key groups across all six combinations

duplicate_df = pd.DataFrame([
    {
        "comparison": group["comparison"],
        "group_id": group["group_id"],
        "group_size": group["group_size"],
        "label_hashes_all_equal": group["label_hashes_all_equal"],
        "members": group["members"],
    }
    for group in results["duplicate_groups"]
])

print("Duplicate groups by comparison:")
if not duplicate_df.empty:
    display(
        duplicate_df.groupby("comparison").agg(
            duplicate_groups=("group_id", "count"),
            largest_group=("group_size", "max")
        ).reset_index()
    )
    display(duplicate_df)
else:
    print("No repeated pair-key groups found.")


In [ ]:
# 5. Inspect partial matches
# These are cases where imageA repeats but imageB differs, or vice versa.

partial_df = pd.DataFrame(results["partial_matches"])

if not partial_df.empty:
    display(
        partial_df.groupby(
            ["comparison", "match_type"]
        ).size().reset_index(name="group_count")
    )
    display(partial_df)
else:
    print("No partial image matches found.")


In [ ]:
# 6. Inspect the train manifest and its drop reasons

manifest_path = OUTPUT_DIR / "train_manifest.csv"
manifest_df = pd.read_csv(manifest_path)

print("Manifest rows:", len(manifest_df))
print("\nKeep/drop counts:")
display(manifest_df["keep"].value_counts(dropna=False))

print("\nDrop reasons:")
display(
    manifest_df.loc[~manifest_df["keep"]]
    ["reason"].value_counts()
    .rename_axis("reason")
    .reset_index(name="rows")
)

print("\nFirst 10 manifest rows:")
display(manifest_df.head(10))

assert len(manifest_df) == 7120
assert int(manifest_df["keep"].sum()) + int((~manifest_df["keep"]).sum()) == 7120


In [ ]:
# 7. Inspect validation/test duplicate report (report-only)

val_test_report_path = OUTPUT_DIR / "dup_report_val_test.csv"
val_test_df = pd.read_csv(val_test_report_path)

print("Validation/test duplicate report rows:", len(val_test_df))
display(val_test_df)


In [ ]:
# 8. Short paste-back summary

summary = results["manifest_summary"]

print("\n========== GEOCHANGE AI — STEP 2A.2 SUMMARY ==========")
print("Original train rows:", summary["train_rows"])
print("Rows kept:", summary["kept"])
print("Rows dropped:", summary["dropped"])
print("Dropped by reason:", summary["dropped_by_reason"])
print("Kept + dropped:", summary["kept"] + summary["dropped"])
print("Any duplicate group with >2 members:", results["any_group_over_two"])
print("Number of groups with >2 members:", len(results["groups_over_two"]))
print("Val/test rows dropped: 0 (report only)")
print("Original Parquet files modified: NO")
print("Manifest:", manifest_path)
print("Val/test duplicate report:", val_test_report_path)
print("======================================================")
